# Continue ten varied pilots, adapt to tactical, then collect NN duels

**Run All performs the whole campaign:** load the ten completed **91-input** pilots from notebook 30; train each for **200 additional epochs against simple**; assess every pilot against tactical; train each for **300 additional tactical epochs**; repeat the tactical assessment; freeze the adapted population; collect and verify **10,000 NN-versus-NN physical flights** (20,000 aircraft perspectives).

Each pilot retains its saved Transformer shape, PPO value/entropy coefficients, selected-skill parameter contract and hybrid reward recipe. We load the validation-selected policy **and critic** at each stage boundary with fresh Adam, as in notebook 07. Initial weights remain eligible for selection on the new stage's validation set. Within a stage, a restart restores the current policy, optimizer, scaler, RNGs and completed epoch. The corpus uses the **tactical-selected** checkpoints.

The final data follows notebook 11's observations/tracks/target-decisions structure, using the current two-skill catalogue and **all 91 actual policy features at 10 Hz**, decisions at 1 Hz, both aircraft, and up to 300 seconds or first defeat per flight. Every input history is reconstructible and checked against the input used for inference. No Git publication is launched by this notebook.

## Configure once, then Run All

Install in this notebook kernel's environment from the repository root, then restart the kernel:

```bash
python -m pip install -e '.[ml,video]'
```

For CUDA, use a compatible CUDA-enabled PyTorch installation. `BVR_WEEKEND_SOURCE` defaults to `artifacts/rl_pilot_simplified/varied_transformers/20261112T232045_249258Z` beneath this checkout, the batch identified by notebook 30's completed training output. The setup cell sets this environment variable automatically. Override it (or `SOURCE` below) if you copied the batch elsewhere. The batch must contain `pilots.json` and all ten pilot directories. These trained files are local artifacts and are not included in Git; run on the machine holding them or copy them to the configured location. Later Run All executions reuse the source and progress pinned to `JOB`.

**Keep JOB and all settings unchanged to resume.** A different experiment needs a new JOB and OUTPUT. Missing checkpoints, changed source/configuration/software, incompatible catalogues and invalid devices fail during preflight, before the long training sequence. Keep the machine awake and the kernel running. The notebook reports progress every 30 seconds and writes full subprocess logs to disk.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import signal
import subprocess
import sys
import time

root_override = os.getenv("BVR_WEEKEND_ROOT")
candidates = [Path(root_override).expanduser()] if root_override else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p.resolve() for p in candidates if (p / "scripts/run_varied_pilot_weekend.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_WEEKEND_ROOT to the Trajectory_Classification checkout.")

# Notebook 30's completed training output identifies this batch. Resolve it
# relative to this checkout so the same setting works on Windows and Linux.
DEFAULT_SOURCE = ROOT / "artifacts/rl_pilot_simplified/varied_transformers/20261112T232045_249258Z"
os.environ["BVR_WEEKEND_SOURCE"] = os.getenv("BVR_WEEKEND_SOURCE") or str(DEFAULT_SOURCE)
SOURCE = Path(os.environ["BVR_WEEKEND_SOURCE"]).expanduser().resolve()
JOB = Path(os.getenv("BVR_WEEKEND_JOB", str(ROOT / "artifacts/rl_pilot_simplified/varied_weekend_v1"))).expanduser().resolve()
OUTPUT = Path(os.getenv("BVR_WEEKEND_OUTPUT", str(ROOT / "shared/corpora/nn-duels-varied-v1" / JOB.name))).expanduser().resolve()
SIMPLE_EPOCHS = int(os.getenv("BVR_WEEKEND_SIMPLE_EPOCHS", "200"))
TACTICAL_EPOCHS = int(os.getenv("BVR_WEEKEND_TACTICAL_EPOCHS", "300"))
ASSESSMENT_SCENARIOS = int(os.getenv("BVR_WEEKEND_ASSESSMENT_SCENARIOS", "300"))
FLIGHTS = int(os.getenv("BVR_WEEKEND_FLIGHTS", "10000"))
FLIGHT_SECONDS = float(os.getenv("BVR_WEEKEND_FLIGHT_SECONDS", "300"))
CORPUS_SEED = int(os.getenv("BVR_WEEKEND_CORPUS_SEED", "800000000"))
WORKERS = int(os.getenv("BVR_WEEKEND_WORKERS", str(max(1, min(32, int((os.cpu_count() or 1) * .8))))))
TORCH_THREADS = int(os.getenv("BVR_WEEKEND_TORCH_THREADS", str(min(8, os.cpu_count() or 1))))
DEVICE = os.getenv("BVR_WEEKEND_DEVICE", "auto")
INFERENCE_BACKEND = os.getenv("BVR_WEEKEND_INFERENCE_BACKEND", "auto")
SHARD_FLIGHTS = int(os.getenv("BVR_WEEKEND_SHARD_FLIGHTS", "100"))
BENCHMARK_FLIGHTS = int(os.getenv("BVR_WEEKEND_BENCHMARK_FLIGHTS", "32"))
RUN = os.getenv("BVR_WEEKEND_RUN", "1") == "1"
SCRIPT = ROOT / "scripts/run_varied_pilot_weekend.py"
JOB.mkdir(parents=True, exist_ok=True)

ARGS = [sys.executable, "-u", str(SCRIPT), "--job", str(JOB), "--output", str(OUTPUT),
        "--simple-epochs", str(SIMPLE_EPOCHS), "--tactical-epochs", str(TACTICAL_EPOCHS),
        "--assessment-scenarios", str(ASSESSMENT_SCENARIOS), "--flights", str(FLIGHTS),
        "--flight-seconds", str(FLIGHT_SECONDS), "--corpus-seed", str(CORPUS_SEED),
        "--workers", str(WORKERS), "--torch-threads", str(TORCH_THREADS),
        "--inference-backend", INFERENCE_BACKEND, "--shard-flights", str(SHARD_FLIGHTS),
        "--benchmark-flights", str(BENCHMARK_FLIGHTS)]
if SOURCE is not None:
    ARGS += ["--source", str(SOURCE)]
if DEVICE != "auto":
    ARGS += ["--device", DEVICE]
# Optional short-run overrides. Unset values preserve the source training recipe.
for variable, option in {
    "BVR_WEEKEND_EPISODE_SECONDS": "--episode-duration-s",
    "BVR_WEEKEND_SCENARIOS": "--scenarios-per-epoch",
    "BVR_WEEKEND_EVALUATION_SCENARIOS": "--evaluation-scenarios-per-epoch",
    "BVR_WEEKEND_UPDATE_EPOCHS": "--update-epochs",
    "BVR_WEEKEND_EVALUATION_INTERVAL": "--evaluation-interval",
}.items():
    if os.getenv(variable) is not None:
        ARGS += [option, os.environ[variable]]

print(f"Kernel: {sys.executable}\nJOB: {JOB}\nOUTPUT: {OUTPUT}")
print(f"10 pilots x ({SIMPLE_EPOCHS} simple + {TACTICAL_EPOCHS} tactical) additional epochs; {FLIGHTS:,} physical duels.")

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf8"))


def run_logged(arguments, label):
    logs = JOB / "logs"
    logs.mkdir(exist_ok=True)
    log = logs / f"{label}_{time.time_ns()}.log"
    environment = os.environ.copy()
    for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        environment[key] = "1"
    environment["MLFLOW_DISABLE_AGENT_HINT"] = "1"
    print(f"Starting {label}. Full log: {log}", flush=True)
    with log.open("w", encoding="utf8") as stream:
        options = {"start_new_session": True} if os.name != "nt" else {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP}
        process = subprocess.Popen(arguments, cwd=ROOT, env=environment, stdout=stream, stderr=subprocess.STDOUT, **options)
        last_report = time.monotonic()
        try:
            while process.poll() is None:
                time.sleep(2)
                if time.monotonic() - last_report >= 30:
                    status = read_json(JOB / "status.json") if (JOB / "status.json").exists() else {"stage": label}
                    with log.open("rb") as reader:
                        reader.seek(max(0, log.stat().st_size - 4096))
                        lines = reader.read().decode("utf8", errors="replace").splitlines()
                    print(f"{status['stage']} | {lines[-1][:500] if lines else 'starting'}", flush=True)
                    last_report = time.monotonic()
        except BaseException:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"], check=False, capture_output=True)
            else:
                os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if process.returncode:
        with log.open("rb") as reader:
            reader.seek(max(0, log.stat().st_size - 6000))
            tail = reader.read().decode("utf8", errors="replace")
        raise RuntimeError(f"{label} exited with {process.returncode}. Log: {log}\n{tail}")
    print(f"Completed {label}. Log: {log}", flush=True)
    return log


PREFLIGHT_LOG = run_logged(ARGS + ["--preflight-only"], "preflight")
plan = read_json(JOB / "campaign.json")
SOURCE = Path(plan["source"])
print("Pinned notebook-30 source:", SOURCE)
print("Device:", plan["device"], "| simulator workers:", plan["workers"])
# Native libraries stay in child processes until the preflight succeeds.
import pandas as pd
import numpy as np
from IPython.display import display
roster = []
for record in read_json(SOURCE / "pilots.json")["pilots"]:
    metadata = read_json(SOURCE / record["output_dir"] / "config.json")
    training = metadata["training"]
    roster.append({"pilot": record["pilot_id"], **{k: training[k] for k in
                   ("epochs", "episode_duration_s", "scenarios_per_epoch", "evaluation_scenarios_per_epoch",
                    "hidden_size", "transformer_heads", "transformer_layers", "value_coefficient", "entropy_coefficient")}})
display(pd.DataFrame(roster))

## Run the full sequence

This cell blocks until all stages finish, and rerunning it resumes the same campaign. It performs all 200 simple epochs for all ten pilots before the first tactical assessment; all starting tactical assessments finish before tactical training begins. Training/validation seeds are separate from the matched tactical assessment and final NN corpus seeds.

The original learning rate, PPO update passes, batch size and loss/reward settings are inherited from each saved pilot unless an explicit small-run override was supplied. CPU simulator processes remain warm across epochs; PPO uses CUDA when available. The collection stage benchmarks local CPU inference against architecture-grouped CUDA inference when a GPU is available, then uses the measured faster backend. There are 100 ordered pilot pairings, including self-pairs; the default 10,000 flights assigns 100 fresh geometries to each pair.

Each completed epoch commits a full restart checkpoint atomically. Each completed flight commits checksummed Parquet files before its completion marker. Lost process batches retry with their original conditions; persistent failures leave the saved progress and traceback for diagnosis. The final packaging pass writes immutable shards and verifies every observation, physical track, target decision and policy-input history.

In [ ]:
CAMPAIGN_LOG = run_logged(ARGS, "campaign") if RUN else None
if not RUN:
    print("RUN=False: preflight only. Set RUN=True and rerun this cell to start/resume.")
if (JOB / "status.json").exists():
    print(read_json(JOB / "status.json"))

## Inspect training and tactical outcomes

The paired before/after assessment uses the tactical opponent and the same separate 300 scenarios per pilot. It does not select checkpoints. The assessment is a development comparison after inspection, rather than an untouched generalization test. Survival and confirmed elimination can overlap. Each pilot's own reward selects checkpoints, so compare raw reward and PPO loss within a pilot.

In [ ]:
progress_rows = []
for stage, target_epochs in (("simple", SIMPLE_EPOCHS), ("tactical", TACTICAL_EPOCHS)):
    for row in roster:
        directory = JOB / stage / row["pilot"]
        metric_file = directory / "training_metrics.jsonl"
        metrics = pd.read_json(metric_file, lines=True) if metric_file.exists() else pd.DataFrame()
        progress_rows.append({"stage": stage, "pilot": row["pilot"], "completed_epochs":
                              int(metrics.epoch.max()) if len(metrics) else 0, "target_epochs": target_epochs,
                              "selected_checkpoint": (directory / "manifest.json").exists()})
display(pd.DataFrame(progress_rows))

outcomes = []
for phase in ("before_tactical", "after_tactical"):
    for file in sorted((JOB / phase).glob("*/evaluation.json")):
        summary = read_json(file)["summary"]
        outcomes.append({"phase": phase, "pilot": file.parent.name, **{k: summary[k] for k in
                         ("scenarios", "survival_rate", "elimination_rate", "clean_win_rate", "mutual_destruction_rate")}})
if outcomes:
    outcome_table = pd.DataFrame(outcomes)
    display(outcome_table)
    display(outcome_table.groupby("phase")[["survival_rate", "elimination_rate", "clean_win_rate"]].mean())
else:
    print("No tactical assessments saved yet.")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(14, 8), squeeze=False)
for column, stage in enumerate(("simple", "tactical")):
    for row in roster:
        file = JOB / stage / row["pilot"] / "training_metrics.jsonl"
        if not file.exists():
            continue
        metrics = pd.read_json(file, lines=True)
        fresh = metrics[metrics.evaluation_performed.eq(1)]
        axes[0, column].plot(fresh.epoch, fresh.mean_return, label=row["pilot"])
        axes[1, column].plot(metrics.epoch, metrics.loss, label=row["pilot"])
    axes[0, column].set(title=f"{stage}: own validation reward", xlabel="Additional epoch", ylabel="Return")
    axes[1, column].set(title=f"{stage}: PPO objective", xlabel="Additional epoch", ylabel="Loss")
    for axis in axes[:, column]:
        axis.grid(alpha=.2)
if axes[0, 0].lines:
    axes[0, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

## Read the final 91-feature corpus

`OUTPUT/A01/` and `OUTPUT/B01/` each contain **observations**, **tracks**, and **target_decisions** Parquet shards. `episodes.parquet` indexes two perspectives per physical flight; keep both perspectives in the same dataset split using `scenario_group`.

- `obs_00`–`obs_46`: compact own/enemy aircraft state and four friendly missile slots.
- `obs_47`–`obs_86`: four nearest live hostile missiles targeting the observer; presence, relative NWU position/velocity, range, closing speed, seeker state.
- `obs_87`–`obs_90`: own kinetic/potential and target kinetic/potential energy features.

`schema.json` names every feature and its divisor/formula. These are scaled simulator-truth inputs, including threats before seeker activation, matching notebook 30. Missing missile slots are zero. The chronological 20-frame policy history repeats the reset frame on the left. Decisions occur before the next physics step; the final terminal frame has no decision.

**target_decisions describes the other aircraft.** Its `policy_history_sha256` hashes that target's own recorded observation history, available in the opposite perspective. The verifier checks these hashes against every history actually submitted to inference. `tactical_reproduction.zip` contains the exact code, simulator assets, ten adapted checkpoints/configurations, and the source/training provenance. Existing forecasting loaders fixed to 51 inputs need to be configured for these 91 columns.

In [ ]:
if (OUTPUT / "manifest.json").exists():
    manifest = read_json(OUTPUT / "manifest.json")
    schema = read_json(OUTPUT / "schema.json")
    display(pd.Series({k: manifest[k] for k in
                       ("complete", "requested_flights", "physical_flights", "perspective_episodes")}))
    assert len(schema["observation_columns"]) == 91
    display(pd.DataFrame(schema["features"]))
    episodes = pd.read_parquet(OUTPUT / "episodes.parquet")
    if len(episodes):
        display(episodes.groupby(["observer_pilot_id", "pilot_id"]).agg(
            flights=("episode_id", "count"), observer_survival=("observer_survived", "mean"),
            mean_duration_s=("duration_s", "mean")))
        example = episodes.iloc[0]
        def episode_table(side, part):
            return pd.read_parquet(OUTPUT / side / part / f"{example.shard}.parquet",
                                   filters=[("episode_id", "==", example.episode_id)]).sort_values("step")
        target_frames = episode_table(example.target_id, "observations")[schema["observation_columns"]].to_numpy(dtype="<f4")
        decision = episode_table(example.observer_id, "target_decisions").iloc[0]
        indices = np.maximum(0, np.arange(int(decision.step) - 19, int(decision.step) + 1))
        policy_input = target_frames[indices]
        assert policy_input.shape == (20, 91)
        assert hashlib.sha256(policy_input.tobytes()).hexdigest() == decision.policy_history_sha256
        print("Example reconstructed policy input:", policy_input.shape, "| input checksum verified")
else:
    print("Final corpus will appear after collection and packaging finish:", OUTPUT)

## Resume or run a short installation check

After an interruption, restart the kernel and Run All with the same JOB, source, settings and software. Completed pilots/assessments/flights are reused; an interrupted pilot resumes after its last committed epoch. No manual lock removal is needed after the process tree exits. `JOB/failure.json`, `JOB/status.json`, each pilot's `training_metrics.jsonl` and `resume.pt`, and `JOB/logs/` provide diagnostic records. Epoch snapshots are crash recovery for the current stage; best weights remain separate validation-selected candidates. Filesystem failures and persistent simulation errors stop the job with a traceback.

For a short **execution check**, choose fresh JOB/OUTPUT directories and set these environment variables before the first cell:

```python
os.environ.update({
    "BVR_WEEKEND_JOB": "/path/to/fresh/smoke-job",
    "BVR_WEEKEND_OUTPUT": "/path/to/fresh/smoke-corpus",
    "BVR_WEEKEND_SIMPLE_EPOCHS": "1", "BVR_WEEKEND_TACTICAL_EPOCHS": "1",
    "BVR_WEEKEND_EPISODE_SECONDS": "1", "BVR_WEEKEND_SCENARIOS": "3",
    "BVR_WEEKEND_EVALUATION_SCENARIOS": "3", "BVR_WEEKEND_UPDATE_EPOCHS": "1",
    "BVR_WEEKEND_ASSESSMENT_SCENARIOS": "3", "BVR_WEEKEND_FLIGHTS": "3",
    "BVR_WEEKEND_FLIGHT_SECONDS": "3", "BVR_WEEKEND_WORKERS": "1",
    "BVR_WEEKEND_TORCH_THREADS": "1", "BVR_WEEKEND_DEVICE": "cpu",
    "BVR_WEEKEND_INFERENCE_BACKEND": "central_cpu",
})
```

This still requires all ten notebook-30 checkpoints and checks every stage; it does not establish learning quality. Clear the small-run overrides and choose fresh JOB/OUTPUT directories for the full weekend campaign. The full 5,000 additional training epochs and final collection may exceed one weekend on slower hardware; use saved epoch timings to assess throughput. Preserve `resume.pt` until a stage is complete and back up the campaign/output directories to your normal storage.